In [ ]:
# =====================================================================
#  CUSTOMER / SALES CLUSTERING  -  K-Means (with the Elbow method) + Hierarchical
#  Dataset: sales_data_sample.csv  (Kaggle: sample sales data, 2823 order lines)
#  Task: implement K-Means / hierarchical clustering and choose the number
#        of clusters using the elbow method.
#  Run in Google Colab / Jupyter (each "# %%" marks one cell)
# =====================================================================
#
# ---------------------------------------------------------------------
#  PART A: HIGH-LEVEL IDEA (read this first)
# ---------------------------------------------------------------------
# PROBLEM:
#   We have thousands of sales order lines (quantity, price, sales amount, product line,
#   deal size ...). We do NOT have any "answer" column. We want the computer to find
#   GROUPS of similar orders by itself. This is UNSUPERVISED learning, and the task is called
#   CLUSTERING. (Compare: KNN / SVM / neural network earlier were SUPERVISED, they had
#   an answer column to learn from. Here there is no y, and no train/test split.)
#
# HOW K-MEANS WORKS:
#   1. Choose K = number of groups we want.
#   2. Put K "centres" (centroids) at random places.
#   3. ASSIGN every order to its nearest centre  -> K groups.
#   4. MOVE each centre to the average position of its group.
#   5. Repeat steps 3-4 until the centres stop moving.
#   The result: every order gets a cluster number (0, 1, 2, ...).
#
# HOW TO PICK K - THE ELBOW METHOD:
#   For each K we run K-Means and measure INERTIA = the sum of squared distances from every
#   point to its own cluster centre (smaller = tighter clusters).
#   More clusters ALWAYS give a smaller inertia (with K = number of points it becomes 0), so we
#   do not pick the smallest. We plot inertia against K and look for the ELBOW: the point
#   where the curve stops dropping steeply and flattens. Before the elbow, adding a cluster
#   helps a lot; after it, the gain is small, so the elbow K is the best trade-off.
#
# HOW HIERARCHICAL CLUSTERING WORKS (second method, optional in the assignment):
#   Start with every order as its own cluster, then repeatedly MERGE the two closest
#   clusters until one big cluster remains. The merging history drawn as a tree is a
#   DENDROGRAM; cutting the tree at a chosen height gives the clusters.
#
# ---------------------------------------------------------------------
#  PART B: THE ALGORITHM (what we will do, step by step)
# ---------------------------------------------------------------------
#  1. READ the file into a table.
#  2. DROP columns that are not useful for grouping (names, phones, addresses, ids, dates).
#  3. CONVERT the text columns we keep (PRODUCTLINE, DEALSIZE) into 0/1 number columns.
#  4. SCALE all columns to the same range (distance-based methods need this).
#  5. ELBOW METHOD: run K-Means for K = 1..10, plot the inertia, choose K at the elbow.
#  6. RUN K-Means with the chosen K and look at the cluster sizes and what each cluster means.
#  7. VISUALIZE the clusters in 2-D using PCA.
#  8. RUN hierarchical clustering (dendrogram) and compare it with K-Means.
# ---------------------------------------------------------------------
 
 
# %% STEP 0: IMPORT LIBRARIES
import numpy as np                       # maths on arrays
import pandas as pd                      # tables (DataFrame)
import matplotlib.pyplot as plt          # basic plots
import seaborn as sns                    # prettier plots (scatter with colours)
 
from sklearn.preprocessing import StandardScaler          # puts all columns on the same scale
from sklearn.cluster import KMeans, AgglomerativeClustering
# KMeans                  : the K-Means algorithm
# AgglomerativeClustering : hierarchical clustering that merges clusters from the bottom up
from sklearn.decomposition import PCA                     # squeezes many columns into 2 for plotting
from sklearn.metrics import silhouette_score              # quality score of a clustering (-1 to 1)
from scipy.cluster.hierarchy import linkage, dendrogram   # builds and draws the merge tree
 
 
# %% STEP 1: READ THE DATASET
df = pd.read_csv('sales_data_sample.csv', encoding='Latin-1')
# encoding='Latin-1' : this file contains accented characters (e.g. in customer names) that the
# default UTF-8 reader cannot decode and would crash with a UnicodeDecodeError.
print(df.shape)                          # (rows, columns) -> expect (2823, 25)
print(df.head())                         # first 5 rows
print(df.isnull().sum())                 # missing values per column
# ADDRESSLINE2, STATE and POSTALCODE have many missing values, but we do not need them anyway.
 
 
# %% STEP 2: DROP COLUMNS WE DO NOT NEED
cols_to_drop = ['ORDERNUMBER', 'ORDERDATE',                    # an id and a date (MONTH_ID/YEAR_ID already hold the date)
                'STATUS', 'PRODUCTCODE',                        # status label; product code is just a name, its number has no meaning
                'CUSTOMERNAME', 'PHONE', 'CONTACTFIRSTNAME', 'CONTACTLASTNAME',   # personal details
                'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE',    # address (many missing values)
                'COUNTRY', 'TERRITORY']                         # 19 countries would add too many columns
df = df.drop(columns=cols_to_drop)
# .drop(columns=[...]) removes those columns. K-Means measures distances between NUMBERS, so
# names, phones and addresses cannot be used; ids and codes would add fake distances.
print(df.columns.tolist())               # what is left: numbers + PRODUCTLINE and DEALSIZE (text)
 
 
# %% STEP 3: CONVERT THE REMAINING TEXT COLUMNS TO NUMBERS
df = pd.get_dummies(df, columns=['PRODUCTLINE', 'DEALSIZE'], dtype=int)
# pd.get_dummies = "one-hot encoding": replaces a text column by one 0/1 column per category,
# e.g. PRODUCTLINE -> PRODUCTLINE_Classic Cars, PRODUCTLINE_Motorcycles, ...
# (We keep ALL categories here, no drop_first, because for distance-based methods leaving one
# out would make that category look artificially different.)
# dtype=int makes the new columns 0/1 numbers instead of True/False.
print(df.shape)                          # now all columns are numeric
print(df.dtypes.value_counts())
 
 
# %% STEP 4: SCALE THE DATA (mean 0, standard deviation 1)
scaler = StandardScaler()                # create the scaler
X = scaler.fit_transform(df)
# fit_transform learns each column's mean and std, then applies  z = (value - mean) / std.
# WHY: SALES is in the thousands while QTR_ID is 1-4. Without scaling, SALES would completely
# dominate every distance and the clusters would be based on SALES alone (the reference
# notebook had this problem). X is now a plain NumPy array of shape (2823, number of columns).
# There is NO train/test split in clustering: there are no answers to test against.
print("X shape:", X.shape)
 
 
# %% STEP 5: ELBOW METHOD - CHOOSE THE NUMBER OF CLUSTERS
K_values = list(range(1, 11))            # try K = 1, 2, ..., 10
inertias = []                            # inertia for each K
for k in K_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    # n_clusters = K. n_init=10 : run the algorithm 10 times from different random start
    # centres and keep the best run (random starts can give bad results).
    # random_state=42 : repeatable results.
    km.fit(X)                            # fit() runs K-Means on all the data
    inertias.append(km.inertia_)         # .inertia_ = sum of squared distances to the own centre
 
# --- Find the elbow automatically (so you also have a number, not only a picture) ---
# Idea: draw a straight line from the first point (K=1) to the last point (K=10) of the curve.
# The elbow is the point FARTHEST from that straight line.
x_norm = (np.array(K_values) - 1) / (len(K_values) - 1)                # K scaled to 0..1
y_norm = (np.array(inertias) - min(inertias)) / (max(inertias) - min(inertias))   # inertia scaled to 0..1
# After scaling, the straight line from first to last point is x + y = 1, so the distance of
# every point from it is proportional to |x + y - 1|.
elbow_k = K_values[int(np.argmax(np.abs(x_norm + y_norm - 1)))]
# np.abs = absolute value; np.argmax = POSITION of the largest value; we use that position to
# read the matching K.
print("Inertia for each K:", [round(i) for i in inertias])
print("Elbow found at K =", elbow_k)
 
# --- Silhouette score as a second opinion (needs at least 2 clusters) ---
sil_scores = []
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    # fit_predict = fit() and then return the cluster number of every row, in one call.
    sil_scores.append(silhouette_score(X, labels))
    # silhouette_score: for each point compares "how close am I to my own cluster" with
    # "how close am I to the nearest other cluster". Ranges -1 to +1; HIGHER = better separated.
 
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(K_values, inertias, 'bx-')                                # 'bx-' = blue line with x markers
axes[0].axvline(elbow_k, color='red', linestyle='--', label=f'elbow K={elbow_k}')
# axvline draws a vertical line at x = elbow_k.
axes[0].set_xlabel('Number of clusters K'); axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow method'); axes[0].legend()
axes[1].plot(range(2, 11), sil_scores, 'go-')
axes[1].set_xlabel('Number of clusters K'); axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score (higher is better)')
plt.tight_layout(); plt.show()
# HOW TO READ: on the left, find where the curve bends from steep to flat. The red line is the
# automatic guess; if the picture shows a clearer bend somewhere else, use that K instead.
# The right plot is only a cross-check.
 
K = elbow_k
# This is the number of clusters used from here on. You may overwrite it, e.g. K = 3, after
# looking at the elbow picture.
 
 
# %% STEP 6: RUN K-MEANS WITH THE CHOSEN K
kmeans = KMeans(n_clusters=K, n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X)
# fit_predict gives each order a cluster number 0..K-1; we store it as a new column 'Cluster'
# in the (unscaled) table so we can read the real values later.
print("\nCluster sizes:")
print(df['Cluster'].value_counts().sort_index())
# value_counts() = how many orders in each cluster; sort_index() orders by cluster number.
 
# What does each cluster mean? Average of the REAL (unscaled) values per cluster.
profile = df.groupby('Cluster')[['QUANTITYORDERED', 'PRICEEACH', 'SALES', 'MSRP']].mean().round(1)
# groupby('Cluster') splits the rows by cluster number; [[columns]] chooses the columns;
# .mean() gives one average per cluster and column.
print("\nCluster profile (average values):")
print(profile)
# Example reading: the cluster with the highest SALES and QUANTITYORDERED = "big orders";
# the one with the lowest = "small orders".
 
 
# %% STEP 7: VISUALIZE THE CLUSTERS IN 2-D WITH PCA
# We have ~19 columns but a screen has only 2 axes. PCA (Principal Component Analysis)
# combines the columns into 2 new columns (PCA1, PCA2) that keep as much of the information
# (variation) as possible, so the cluster shapes stay visible.
pca = PCA(n_components=2)                # keep 2 new columns
X_2d = pca.fit_transform(X)              # learn the combination and apply it -> shape (2823, 2)
centers_2d = pca.transform(kmeans.cluster_centers_)
# cluster_centers_ = the K centroids (in scaled space). pca.transform puts them into the same
# 2-D picture as the points (transform only: it must use the SAME PCA that was already learned).
 
plt.figure(figsize=(9, 6))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=df['Cluster'], palette='tab10', s=25)
# hue colours every point by its cluster number; palette is the colour set.
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], color='black', marker='x', s=200, label='centroids')
plt.xlabel('PCA1'); plt.ylabel('PCA2'); plt.title(f'K-Means clusters (K={K})'); plt.legend()
plt.show()
 
 
# %% STEP 8: HIERARCHICAL CLUSTERING (second method) AND COMPARISON
Z = linkage(X, method='ward')
# linkage builds the merge history (the tree). method='ward' merges, at each step, the two
# clusters whose union increases the total within-cluster spread the least. This gives compact,
# similar-sized clusters and is the same idea as K-Means.
plt.figure(figsize=(10, 5))
dendrogram(Z, truncate_mode='lastp', p=20)
# dendrogram draws the tree. truncate_mode='lastp', p=20 shows only the LAST 20 merges
# (otherwise 2823 leaves would be unreadable).
plt.xlabel('Clusters (number of orders in brackets)'); plt.ylabel('Merge distance')
plt.title('Dendrogram (truncated)'); plt.show()
# HOW TO READ: the height of a branch = how different the two merged clusters were. Cutting the
# tree with a horizontal line just above a long vertical gap gives the natural number of clusters.
 
hier = AgglomerativeClustering(n_clusters=K, linkage='ward')
hier_labels = hier.fit_predict(X)
# "Agglomerative" = bottom-up merging. n_clusters=K cuts the tree so exactly K clusters remain.
print("\nHierarchical cluster sizes:")
print(pd.Series(hier_labels).value_counts().sort_index())
 
print("\nK-Means (rows) vs Hierarchical (columns):")
print(pd.crosstab(df['Cluster'], hier_labels))
# pd.crosstab counts how many orders fall in each pair (K-Means cluster, hierarchical cluster).
# Cluster NUMBERS are arbitrary labels, so look for one big number per row: if each K-Means
# cluster maps mostly onto one hierarchical cluster, the two methods agree.
 
 
# %% STEP 9: CONCLUSION (write in your journal, with YOUR numbers)
# - The elbow method gave K = (your elbow K) clusters: after that point, adding clusters
#   reduced the inertia only slightly.
# - Scaling was necessary; otherwise SALES (thousands) would decide all the clusters.
# - The cluster profile shows what the groups mean (for example big-quantity / high-sales orders
#   versus small orders), which a business can use for targeting.
# - Hierarchical clustering gave similar groups and its dendrogram offers a second way to
#   see the natural number of clusters; but it needs much more memory and time on big data,
#   while K-Means is fast and scales better.
# - K-Means weaknesses: you must choose K, results depend on the random start (so n_init and
#   random_state), and it assumes roughly round, similar-sized clusters.